### Initialize our LLM instance

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

### CHAINS

In [ ]:
# What exactly is a Chain?


# Suppose you want:

# User topic
#     ↓
# Generate explanation
#     ↓
# Generate summary


# That's a fixed workflow.


# Input
#  ↓
# Step 1
#  ↓
# Step 2
#  ↓
# Output


# The supplied sequential-chain material describes exactly this idea: one step's output becomes the next step's input

### 1. Simplest possible chain 
(LCEL - LangChain Expression Language)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple words."
)

chain = prompt | llm

response = chain.invoke({
    "topic": "RAG"
})

print(response.text)


# Flow:

# {"topic": "RAG"}
#         ↓
#  ChatPromptTemplate
#         ↓
#      Gemini
#         ↓
#      AIMessage

### 2. Simple Chain with streaming

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple words."
)

chain = prompt | llm

for chunk in chain.stream({
    "topic": "RAG"
}):
    print(chunk.text, end="", flush=True)


# So:

# invoke() → complete response
# stream() → response chunks

### 3. Chain with multiple steps

In [ ]:
# Suppose:

# Product
#    ↓
# Generate company name
#    ↓
# Generate slogan


# Create two chains:

company_prompt = ChatPromptTemplate.from_template(
    "Generate one company name for a product about {product}."
)

slogan_prompt = ChatPromptTemplate.from_template(
    "Create a slogan for this company: {company_name}"
)

# First:
company_chain = company_prompt | llm

# Second:
slogan_chain = slogan_prompt | llm

# The important question is:

# How do we pass the output of the first into the second?

# That's where Runnable / LCEL composition becomes important.

### 4. LCEL — the important concept

In [ ]:
# LCEL means LangChain Expression Language.



# You have already seen:

# prompt | llm



# The | means:

# output of left
#       ↓
# input of right



# Example:

# chain = prompt | llm



# Think:

# Prompt
#   │
#   ↓
# Gemini



# You can continue:

# chain = prompt | llm | parser



# Now:

# Prompt
#   ↓
# Gemini
#   ↓
# Parser



# This is the modern replacement for many of the older "Chain" classes in the GFG material.

### 5. Sequential chain using LCEL

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


company_prompt = ChatPromptTemplate.from_template(
    "Generate a company name for {product}."
)

slogan_prompt = ChatPromptTemplate.from_template(
    "Create a slogan for {company_name}."
)


company_chain = ( company_prompt | llm | StrOutputParser() )

slogan_chain = ( slogan_prompt | llm | StrOutputParser() )


company_name = company_chain.invoke({
    "product": "AI study assistant"
})

print(company_name)


slogan = slogan_chain.invoke({
    "company_name": company_name
})

print(slogan)


# Flow:

# product
#   ↓
# company_chain
#   ↓
# company_name
#   ↓
# slogan_chain
#   ↓
# slogan

# This is the fundamental idea behind sequential chains.

### 6. Why StrOutputParser()?

In [ ]:
# Gemini normally gives you an AIMessage.

response = llm.invoke("Hello")
print(type(response))  # O/p = <class 'langchain_core.messages.ai.AIMessage'>

# You get an AI message object.



# With:
# StrOutputParser()


# you get:
# AIMessage
#    ↓
# plain string



# So:

# chain = prompt | llm | StrOutputParser()

# is extremely common.

### 7. Passing data between multiple steps
Now we'll make the entire workflow composable.

In [ ]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


company_prompt = ChatPromptTemplate.from_template(
    "Generate a company name for {product}."
)

slogan_prompt = ChatPromptTemplate.from_template(
    "Create a slogan for {company_name}."
)


company_chain = ( company_prompt | llm | StrOutputParser() )

full_chain = (
    company_chain
    | RunnableLambda(
        lambda company_name: {
            "company_name": company_name
        }
    )
    | slogan_prompt
    | llm
    | StrOutputParser()
)

result = full_chain.invoke({
    "product": "AI study assistant"
})

print(result)



# Flow:

# product
#    ↓
# Company Prompt
#    ↓
# Gemini
#    ↓
# company_name
#    ↓
# Slogan Prompt
#    ↓
# Gemini
#    ↓
# slogan

### 8. RunnableLambda Intro (Explained in detailed in step 11)

In [ ]:
# That part is simply converting the previous output into the dictionary 
# format that the next prompt expects.

RunnableLambda(
    lambda company_name: {
        "company_name": company_name
    }
)


# Step by step


# Before it:
# Gemini
#  ↓
# "StudyMate AI"



# So company_name receives:

# company_name = "StudyMate AI"



# Then RunnableLambda runs this function:

# lambda company_name: {
#     "company_name": company_name
# }



# It converts:

# "StudyMate AI"

# into:

# {
#     "company_name": "StudyMate AI"
# }



# Why?

# Because your next prompt has:

# slogan_prompt = ChatPromptTemplate.from_template(
#     "Create a slogan for {company_name}."
# )



# That prompt expects an input with the key:

# "company_name"



# So the flow is:

# company_chain
#      ↓
# "StudyMate AI"
#      ↓
# RunnableLambda
#      ↓
# {"company_name": "StudyMate AI"}
#      ↓
# slogan_prompt
#      ↓
# "Create a slogan for StudyMate AI."
# In one sentence

# RunnableLambda is being used here to transform the company name string into a dictionary with the company_name key required by the next prompt.

### 9. RunnablePassthrough
RunnablePassthrough is one of the most useful LCEL components for keeping the original input while also creating other values from it.

In [ ]:
# Suppose your input is:

# {
#     "product": "AI Study Assistant"
# }

# If you use:

# RunnablePassthrough()

# the input simply passes through:

# "AI Study Assistant"
#         ↓
# RunnablePassthrough
#         ↓
# "AI Study Assistant"

# No modification happens.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

chain = RunnablePassthrough()

result = chain.invoke("Hello Gemini!")

print(result)




# Output: Hello Gemini!
# Nothing changed.

# Flow
# "Hello Gemini!"
#        ↓
# RunnablePassthrough()
#        ↓
# "Hello Gemini!"



# Think:

# RunnablePassthrough
#         ↓
# "Don't change this input"

In [ ]:
# Why do we actually need it?

# Its real usefulness appears with RunnableParallel.



# Suppose you have:

# {
#     "topic": "Artificial Intelligence"
# }



# You want to create two things:

# original topic → keep it
# topic → create explanation


# You can do:


from langchain_core.runnables import RunnableParallel, RunnablePassthrough

chain = RunnableParallel(
    original=RunnablePassthrough(),
    explanation=lambda x: f"Explain {x['topic']}"
)

result = chain.invoke({
    "topic": "Artificial Intelligence"
})

print(result)



# Conceptually:

#                 ┌── original ──→ original input
# input ──────────┤
#                 └── explanation → generated explanation



# The important part is:

# original=RunnablePassthrough()


# It means:

# Put the input here without changing it.

### 10. RunnableParallel
Sometimes you want multiple things to happen from the same input.

In [ ]:
# Sometimes you want multiple things to happen from the same input.

#               Input
#              /     \
#             ↓       ↓
#         Chain A   Chain B
#             ↓       ↓
#          Output A Output B

In [ ]:
from langchain_core.runnables import RunnableParallel


parallel_chain = RunnableParallel(
    explanation=(
        ChatPromptTemplate.from_template(
            "Explain {topic}."
        )
        | llm
        | StrOutputParser()
    ),

    example=(
        ChatPromptTemplate.from_template(
            "Give a simple example of {topic}."
        )
        | llm
        | StrOutputParser()
    )
)

result = parallel_chain.invoke({
    "topic": "RAG"
})

print(result["explanation"])
print(result["example"])

In [ ]:
# Conceptually:

#              RAG
#             /   \
#            ↓     ↓
#     Explanation  Example

# This is useful when different pieces of information can be generated independently.

### 11. RunnableLambda

Use it when you need normal Python logic inside a chain.

In [ ]:
from langchain_core.runnables import RunnableLambda

def uppercase(text):
    return text.upper()

chain = (
    prompt
    | llm
    | StrOutputParser()
    | RunnableLambda(uppercase)
)

# Flow:

# Prompt
#  ↓
# Gemini
#  ↓
# String
#  ↓
# Python function
#  ↓
# Uppercase string

# This becomes useful when connecting AI operations with your own backend logic.

### 12. Example on runnables

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough
)
from langchain_core.output_parsers import StrOutputParser

explanation_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple words."
)

summary_prompt = ChatPromptTemplate.from_template(
    "Give a one-line summary of {topic}."
)

chain = RunnableParallel(
    topic=RunnablePassthrough(),
    explanation=explanation_prompt | llm | StrOutputParser(),
    summary=summary_prompt | llm | StrOutputParser()
)

result = chain.invoke({
    "topic": "Retrieval Augmented Generation"
})

print("Topic:")
print(result["topic"])

print("\nExplanation:")
print(result["explanation"])

print("\nSummary:")
print(result["summary"])


# Flow
#                          ┌── topic ───────→ original topic
#                          │
# input ───────────────────┼── explanation ─→ Gemini
#                          │
#                          └── summary ─────→ Gemini

# This is a very common LCEL pattern.

### 13. Routing / Branching (RunnableBranch)

In [ ]:
# Sometimes:

# User question
#       ↓
# What type?
#    /       \
# coding    general
#   ↓          ↓
# Chain A    Chain B

# You don't always need an agent for this.

# You can use deterministic routing:





# Routing means choosing different chains based on the input.


# Instead of always doing:
# Input → Chain A


# you do:
#                 ┌→ Chain A
# Input → check ──┤
#                 └→ Chain B



# In LangChain, one modern way to do this is RunnableBranch.
# Example: Route questions by type



# Suppose we have two chains:

# If the user asks about Python → Python explanation chain
# Otherwise → General explanation chain

In [ ]:
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

# Python chain
python_prompt = ChatPromptTemplate.from_template(
    "Explain this Python topic in simple words: {question}"
)

python_chain = python_prompt | llm | StrOutputParser()


# General chain
general_prompt = ChatPromptTemplate.from_template(
    "Answer this question in simple words: {question}"
)

general_chain = general_prompt | llm | StrOutputParser()


# Routing
router = RunnableBranch(
    (
        lambda x: "python" in x["question"].lower(),
        python_chain
    ),
    general_chain
)


# Test
result = router.invoke({
    "question": "What is a Python decorator?"
})

print(result)

In [ ]:

# How it works

# The input is:

# {
#     "question": "What is a Python decorator?"
# }



# The condition checks:

# lambda x: "python" in x["question"].lower()



# Since "python" is present:




#                     ┌── Python chain
#                     │
# Question → condition
#                     │
#                     └── General chain

# So the Python chain runs.




# If the input is:
# result = router.invoke({
#     "question": "What is RAG in AI?"
# })




# "python" is not present, so the default chain runs:

# "What is RAG in AI?"
#         ↓
#  condition = False
#         ↓
# general_chain
#         ↓
# Gemini




# The important syntax
# RunnableBranch(
#     (
#         condition,
#         chain_to_run_if_true
#     ),
#     default_chain
# )





# So remember:

# RunnableBranch = if condition → use this chain, otherwise → use another chain. 


## Chain vs Agent

In [1]:
# This distinction is extremely important.



# Chain

# You decide:

# Step 1
#  ↓
# Step 2
#  ↓
# Step 3





# Agent

# You give the model:

# Tools
# +
# Goal



# and the model decides:

# Should I use Tool A?
#        ↓
# Tool A result
#        ↓
# Should I use Tool B?
#        ↓
# Final answer